In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# 출력 예쁘게 하기
from rich.console import Console
from rich.table import Table

console = Console()

def rich_docs(docs, max_len=140, title="Retriever Results"):
    table = Table(title=title)
    table.add_column("#", justify="right")
    table.add_column("Source")
    table.add_column("Page", justify="right")
    table.add_column("Preview")

    for i, d in enumerate(docs, 1):
        m = d.metadata or {}
        src = (m.get("source","") or "").split("/")[-1]
        page = str(m.get("page_label", m.get("page",0)+1))
        text = (d.page_content or "").strip().replace("\n", " ")
        table.add_row(str(i), src, page, (text[:max_len] + ("…" if len(text) > max_len else "")))

    console.print(table)

### retriever 설정값
- 일반 RAG 기본값: similarity or mmr
- 중복이 많을 경우: mmr
- 그 외 필터링이 필요한 경우: search_kwargs 에 다양한 옵션값을 넣어주면 됩니다.

### advanced_retriever
- 길이가 길 경우: compressed_retriever -> 필요할 때 Parent-child
- 용어가 중요할 경우: hybrid(vec + bm25)
- 정확도 극대화: similarity -> reranker -> reorder

In [3]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

In [5]:
emb = OpenAIEmbeddings(model="text-embedding-3-small")
db_path = "../7_vectorstore/chroma_store"
collection_name = "samsung_all"
vectorstore = Chroma(
    collection_name=collection_name,
    persist_directory= db_path,
    embedding_function=emb
)
vectorstore._collection.count()

444

In [6]:
dim_size = len(emb.embed_query("안녕하세요"))
dim_size

1536

### 1. 벡터 기반 검색기(유사도/mmr/score_threshold/filter)

In [ ]:
# 1. 유사도 기반
question = "삼성의 지속가능성에 대해 알려줘"

ret_similarity = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {
        "k":10,
    }
)

result = ret_similarity.invoke(question)
result

[Document(id='samsung_2025::7d968320-9092-4bb0-bdd5-ded00a6615ff', metadata={'creationdate': '2025-07-10T16:11:16+09:00', 'page_label': '86', 'moddate': '2025-09-04T16:51:11+09:00', 'page': 85, 'total_pages': 87, 'producer': 'Adobe PDF Library 15.0', 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'trapped': '/False', 'creator': 'Adobe InDesign 15.1 (Macintosh)'}, page_content='·  삼성전자주식회사 지속가능경영 웹사이트   \nhttp://www.samsung.com/sec/sustainability/main\n·  삼성전자주식회사 IR 웹사이트   \nhttp://www.samsung.com/sec/ir\n·  삼성전자주식회사 뉴스룸   \nhttp://news.samsung.com/kr   \nhttp://news.samsung.com/global\n담당 부서\n· 삼성전자주식회사 지속가능경영추진센터\n· 주소: 16677 경기도 수원시 영통구 삼성로 129(매탄동)\n· 이메일: sustainability.sec@samsung.com\n참고 자료\n·  사업 보고서 \n·  기업지배구조 보고서 \n·  삼성전자 책임광물 관리 보고서 \n·  행동규범 \n·  행동규범 가이드라인 \n미래 예측 진술 공지\n삼성전자주식회사의 지속가능경영보고서에서 삼성전자의 지속가능경영 목표 및  전략과 관련된 \n것을 포함하여 이 루어진 모 든 특정 내용은 관련법상 미래  예측 진술에 해 당할 수 있습니다. 이 \n지속가능경영보고서에서는 향후 상황 및 지속가능경영 성과에 대한 삼성전자의 현재 견해를 반영하는 \n미래 예측 진

In [14]:
rich_docs(result, title="유사도 기반 top5개 확인")

                                              유사도 기반 top5개 확인                                              
┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  # ┃ Source                                           ┃ Page ┃ Preview                                          ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│  1 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ ·  삼성전자주식회사 지속가능경영 웹사이트        │
│    │                                                  │      │ http://www.samsung.com/sec/sustainability/main · │
│    │                                                  │      │ 삼성전자주식회사 IR 웹사이트                     │
│    │                                                  │      │ http://www.samsung.com/sec/ir ·                  │
│    │                                                  │      │ 삼성전자주식회사 뉴…                             │
│  2 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 삼성전자 지속가능경영보고서 2025 86              │
│    │                                                  │      │ 삼성전자주식회사는 경제·사회·환경적 가치 창출    │
│    │                                                  │      │ 성과를 다양한 이해관계자와 투명하게 소통하기     │
│    │                                                  │      │ 위해 2025년 열여덟 번째 지속가능경영보고서를     │
│    │                                                  │      │ 발간합니다. 작성 기준 본 보고서는 지속가능경영   │
│    │                                                  │      │ 보고 기준인 GRI(G…                               │
│  3 │ Sustainability_report_2024_kr.pdf                │   82 │ 비록 삼성전자주식회사는 지속가능경영보고서의     │
│    │                                                  │      │ 미래 예측 진술이 시의성 있고 합리적인 정보, 가 … │
│    │                                                  │      │ 및 믿음에 기반한다고 판단하지만, 이러한 미래     │
│    │                                                  │      │ 예측 진술(그리고 이를 이루는 정보, 가정 및       │
│    │                                                  │      │ 믿음)은  다양한 요인, 리스크, 불확실성의         │
│    │                                                  │      │ 영향권에 있으므로…                               │
│  4 │ Sustainability_report_2024_kr.pdf                │   82 │ ISAE3000을 적용했습니다. 관련 정보 ·             │
│    │                                                  │      │ 삼성전자주식회사 대표 홈페이지                   │
│    │                                                  │      │ http://www.samsung.com/sec ·  삼성전자주식회사   │
│    │                                                  │      │ 지속가능경영 웹사이트                            │
│    │                                                  │      │ http://www.samsung.com/sec/sustainabilit…        │
│  5 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 비록 삼성전자주식회사는 지속가능경영보고서의     │
│    │                                                  │      │ 미래 예측 진술이 시의성 있고 합리적인 정보, 가 … │
│    │                                                  │      │ 및 믿음에 기반한다고 판단하지만, 이러한 미래     │
│    │                                                  │      │ 예측 진술(그리고 이를 이루는 정보, 가정 및       │
│    │                                                  │      │ 믿음)은  다양한 요인, 리스크, 불확실성의         │
│    │                                                  │      │ 영향권에 있으므로…                               │
│  6 │ Sustainability_report_2024_kr.pdf                │   55 │ 삼성전자 지속가능경영보고서 2024 55Our Company   │
│    │                                                  │      │ AppendixMateriality Assessment Facts & Figures   │
│    │                                                  │      │ PrinciplePlanet People Facts & Figures 56        │
│    │                                                  │      │ 경제성과 57     사…                              │
│  7 │ Samsung_Electronics_Sustainability_Report_2025_… │    1 │ 삼성전자 지속가능경영보고서 2025 A Journey       │
│    │                                                  │      │ Towards   a Sustainable Future A Journey         │
│    │   

In [23]:
# 2. mmr 기반
question = "삼성의 지속가능성에 대해 알려줘"

ret_mmr = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {
        "k":5,
        "fetch_k": 20,
        "lambda_mult": 0.5, # 1에 가까울수록 내용기반, 0에 가까울수록 키워드 기반,
        "filter": {"source": '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf'}
    }
)

result = ret_mmr.invoke(question)
result

[Document(id='samsung_2025::7d968320-9092-4bb0-bdd5-ded00a6615ff', metadata={'page_label': '86', 'creationdate': '2025-07-10T16:11:16+09:00', 'total_pages': 87, 'source': '../../data/Samsung_Electronics_Sustainability_Report_2025_KOR.pdf', 'moddate': '2025-09-04T16:51:11+09:00', 'producer': 'Adobe PDF Library 15.0', 'trapped': '/False', 'page': 85, 'creator': 'Adobe InDesign 15.1 (Macintosh)'}, page_content='·  삼성전자주식회사 지속가능경영 웹사이트   \nhttp://www.samsung.com/sec/sustainability/main\n·  삼성전자주식회사 IR 웹사이트   \nhttp://www.samsung.com/sec/ir\n·  삼성전자주식회사 뉴스룸   \nhttp://news.samsung.com/kr   \nhttp://news.samsung.com/global\n담당 부서\n· 삼성전자주식회사 지속가능경영추진센터\n· 주소: 16677 경기도 수원시 영통구 삼성로 129(매탄동)\n· 이메일: sustainability.sec@samsung.com\n참고 자료\n·  사업 보고서 \n·  기업지배구조 보고서 \n·  삼성전자 책임광물 관리 보고서 \n·  행동규범 \n·  행동규범 가이드라인 \n미래 예측 진술 공지\n삼성전자주식회사의 지속가능경영보고서에서 삼성전자의 지속가능경영 목표 및  전략과 관련된 \n것을 포함하여 이 루어진 모 든 특정 내용은 관련법상 미래  예측 진술에 해 당할 수 있습니다. 이 \n지속가능경영보고서에서는 향후 상황 및 지속가능경영 성과에 대한 삼성전자의 현재 견해를 반영하는 \n미래 예측 진

In [ ]:
rich_docs(result, title="mmr 기반 중복 최소화 top5개 확인")

                                               mmr 기반 top5개 확인                                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Source                                           ┃ Page ┃ Preview                                           ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ ·  삼성전자주식회사 지속가능경영 웹사이트         │
│   │                                                  │      │ http://www.samsung.com/sec/sustainability/main ·  │
│   │                                                  │      │ 삼성전자주식회사 IR 웹사이트                      │
│   │                                                  │      │ http://www.samsung.com/sec/ir ·  삼성전자주식회 … │
│   │                                                  │      │ 뉴…                                               │
│ 2 │ Samsung_Electronics_Sustainability_Report_2025_… │    4 │ 삼성전자 지속가능경영보고서 2025 04 Our Company   │
│   │                                                  │      │ AppendixFacts & Figures PrinciplePlanet People    │
│   │                                                  │      │ 주주, 고객, 협력회사, 그리고 임직원 여러분,       │
│   │                                                  │      │ 2024년은 글로벌 지정학적 리스크와 AI 기술의 성장  │
│   │                                                  │      │ …                                                 │
│ 3 │ Samsung_Electronics_Sustainability_Report_2025_… │   76 │ 삼성전자 지속가능경영보고서 2025 76 독립된        │
│   │                                                  │      │ 인증인의 인증보고서 Our Company AppendixFacts &   │
│   │                                                  │      │ Figures PrinciplePlanet People                    │
│ 4 │ Samsung_Electronics_Sustainability_Report_2025_… │   54 │ 삼성전자 지속가능경영보고서 2025 54 Bespoke AI    │
│   │                                                  │      │ 로봇청소기 보안 인증 획득 2024년에는 Bespoke AI   │
│   │                                                  │      │ 스팀 로봇청소기가 KISA 개인정보보호중심설계 (PbD… │
│   │                                                  │      │ Privacy by Design) 인증과 KISA IoT 보안 인증 중 … │
│ 5 │ Samsung_Electronics_Sustainability_Report_2025_… │   51 │ 삼성전자 지속가능경영보고서 2025 51 2024년 운영   │
│   │                                                  │      │ 성과 교육 운영 센터 5개 교육생 2,200명 활동 미래  │
│   │                                                  │      │ 역량 강화를 위한 청소년 교육 삼성전자는           │
│   │                                                  │      │ 청소년들이 미래 사회를 이끌 주역으로서 혁신을     │
│   │                                                  │      │ 주도하며  잠재력을 발휘하여 더 나은 사회를 만들…  │
└───┴──────────────────────────────────────────────────┴──────┴───────────────────────────────────────────────────┘

In [ ]:
# 3. score로 제어  - similarity_score_threshold
question = "삼성의 지속가능성에 대해 알려줘"

ret_score = vectorstore.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {
        "score_threshold": 0.3,
        "k":5,    
    },
)

result = ret_score.invoke(question)
rich_docs(result, title="score로 top5개 확인")


                                                score로 top5개 확인                                                
┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ # ┃ Source                                           ┃ Page ┃ Preview                                           ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ ·  삼성전자주식회사 지속가능경영 웹사이트         │
│   │                                                  │      │ http://www.samsung.com/sec/sustainability/main ·  │
│   │                                                  │      │ 삼성전자주식회사 IR 웹사이트                      │
│   │                                                  │      │ http://www.samsung.com/sec/ir ·  삼성전자주식회 … │
│   │                                                  │      │ 뉴…                                               │
│ 2 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 삼성전자 지속가능경영보고서 2025 86               │
│   │                                                  │      │ 삼성전자주식회사는 경제·사회·환경적 가치 창출     │
│   │                                                  │      │ 성과를 다양한 이해관계자와 투명하게 소통하기 위 … │
│   │                                                  │      │ 2025년 열여덟 번째 지속가능경영보고서를           │
│   │                                                  │      │ 발간합니다. 작성 기준 본 보고서는 지속가능경영    │
│   │                                                  │      │ 보고 기준인 GRI(G…                                │
│ 3 │ Sustainability_report_2024_kr.pdf                │   82 │ 비록 삼성전자주식회사는 지속가능경영보고서의 미 … │
│   │                                                  │      │ 예측 진술이 시의성 있고 합리적인 정보, 가정  및   │
│   │                                                  │      │ 믿음에 기반한다고 판단하지만, 이러한 미래 예측    │
│   │                                                  │      │ 진술(그리고 이를 이루는 정보, 가정 및 믿음)은     │
│   │                                                  │      │ 다양한 요인, 리스크, 불확실성의 영향권에          │
│   │                                                  │      │ 있으므로…                                         │
│ 4 │ Sustainability_report_2024_kr.pdf                │   82 │ ISAE3000을 적용했습니다. 관련 정보 ·              │
│   │                                                  │      │ 삼성전자주식회사 대표 홈페이지                    │
│   │                                                  │      │ http://www.samsung.com/sec ·  삼성전자주식회사    │
│   │                                                  │      │ 지속가능경영 웹사이트                             │
│   │                                                  │      │ http://www.samsung.com/sec/sustainabilit…         │
│ 5 │ Samsung_Electronics_Sustainability_Report_2025_… │   86 │ 비록 삼성전자주식회사는 지속가능경영보고서의 미 … │
│   │                                                  │      │ 예측 진술이 시의성 있고 합리적인 정보, 가정  및   │
│   │                                                  │      │ 믿음에 기반한다고 판단하지만, 이러한 미래 예측    │
│   │                                                  │      │ 진술(그리고 이를 이루는 정보, 가정 및 믿음)은     │
│   │                                                  │      │ 다양한 요인, 리스크, 불확실성의 영향권에          │
│   │                                                  │      │ 있으므로…                                         │
└───┴──────────────────────────────────────────────────┴──────┴───────────────────────────────────────────────────┘